# 第 6 章 ロジスティック回帰

「0 か 1 か」という硬い予測を、シグモイド関数で「0 から 1 の確率」に置き換えます。

対応する記事: [第 6 章 ロジスティック回帰（Kotlin Notebook の言語版）](../../../docs/article/grokking-machine-learning/kotlin/ch06.md)

実装本体: `apps/grokking-ml-kotlin/src/`

## セットアップ

実装本体をビルドした JAR を読み込みます。**ノートブックにコードを複製せず、記事と同じ実装をそのまま使います。**

先に JAR を作っておいてください。

```bash
cd apps/grokking-ml-kotlin
./gradlew jar
```

IntelliJ IDEA の Kotlin Notebook プラグイン、または [Kotlin Jupyter カーネル](https://github.com/Kotlin/kotlin-jupyter) で開きます。

```bash
pip install kotlin-jupyter-kernel
jupyter lab notebooks/
```

In [1]:
@file:DependsOn("../build/libs/grokking-ml-kotlin-0.1.0.jar")

import ch06.*

## シグモイド関数

実数を 0 から 1 の範囲へ押し込む関数です。**大きな負の入力でも壊れない実装** になっていることを確かめます（素朴な `1/(1+exp(-x))` は Python では例外になります）。

In [2]:
listOf(-1000.0, -5.0, -1.0, 0.0, 1.0, 5.0, 1000.0).forEach { x ->
    println("sigmoid(%8.1f) = %.6f".format(x, sigmoid(x)))
}

println()
println("対称性 sigmoid(2) + sigmoid(-2) = " + (sigmoid(2.0) + sigmoid(-2.0)))

sigmoid( -1000.0) = 0.000000
sigmoid(    -5.0) = 0.006693
sigmoid(    -1.0) = 0.268941


sigmoid(     0.0) = 0.500000
sigmoid(     1.0) = 0.731059
sigmoid(     5.0) = 0.993307


sigmoid(  1000.0) = 1.000000

対称性 sigmoid(2) + sigmoid(-2) = 0.9999999999999999


## 対数損失は「確信の度合い」を測る

**当たったかどうかではなく、どれくらいの確信で当たったか** を測ります。0.51 で正解しても損失は 0.67 残るので、学習はまだ進みます。

In [3]:
import kotlin.math.ln

println("%10s %22s".format("予測確率", "正解が 1 のときの損失"))
listOf(0.99, 0.9, 0.51, 0.5, 0.1, 0.01).forEach { probability ->
    println("%10.2f %22.4f".format(probability, -ln(probability)))
}

      予測確率           正解が 1 のときの損失
      0.99                 0.0101


      0.90                 0.1054
      0.51                 0.6733


      0.50                 0.6931


      0.10                 2.3026
      0.01                 4.6052


## 学習

第 5 章と同じデータを使います。**初期の損失 0.6931 は `-log(0.5)`**、つまり「すべて五分五分」の状態です。第 5 章のパーセプトロン誤差が初期状態で 0 だったのと対照的です。

In [4]:
val points = listOf(listOf(1.0, 0.0), listOf(0.0, 2.0), listOf(1.0, 1.0), listOf(1.0, 2.0),
                    listOf(1.0, 3.0), listOf(2.0, 2.0), listOf(2.0, 3.0), listOf(3.0, 2.0))
val labels = listOf(0, 0, 0, 0, 1, 1, 1, 1)

val (trained, losses) = logisticRegression(points, labels, learningRate = 0.1, epochs = 1000, seed = 0)

println("重み   " + trained.weights.map { "%.4f".format(it) })
println("バイアス %.4f".format(trained.bias))
println("損失   %.4f → %.4f".format(losses.first(), losses.last()))
println("正解率 %.2f".format(accuracy(trained, points, labels)))

重み   [2.0201, 1.5903]
バイアス -5.5970
損失   0.6931 → 0.1582
正解率 1.00


## 確率としての出力

単に分類できているだけでなく、**どちらがより確からしいかまで答えられます。**

In [5]:
points.zip(labels).forEach { (point, label) ->
    val probability = trained.predictProbability(point)
    val bar = "#".repeat((probability * 40).toInt())
    println("(%.0f,%.0f) 正解=%d  %.4f %s".format(point[0], point[1], label, probability, bar))
}

(1,0) 正解=0  0.0272 #
(0,2) 正解=0  0.0819 ###
(1,1) 正解=0  0.1206 ####


(1,2) 正解=0  0.4022 ################
(1,3) 正解=1  0.7674 ##############################


(2,2) 正解=1  0.8353 #################################


(2,3) 正解=1  0.9614 ######################################


(3,2) 正解=1  0.9745 ######################################


## 試してみる: 閾値を変える

**確率が手に入ると、再学習せずに判定の厳しさを変えられます。** 第 7 章で扱う適合率と再現率のトレードオフに直結します。

In [6]:
listOf(0.2, 0.5, 0.8).forEach { threshold ->
    val predictions = points.map { trained.predict(it, threshold = threshold) }
    val correct = predictions.zip(labels).count { (p, l) -> p == l }
    println("閾値 %s  予測 %s  正解数 %d/%d".format(threshold, predictions, correct, labels.size))
}

閾値 0.2  予測 [0, 0, 0, 1, 1, 1, 1, 1]  正解数 7/8


閾値 0.5  予測 [0, 0, 0, 0, 1, 1, 1, 1]  正解数 8/8


閾値 0.8  予測 [0, 0, 0, 0, 0, 1, 1, 1]  正解数 7/8
